# 01 — Language Coverage Exploration

Explores `language_codes_comprehensive.csv` to understand the full language target set.

The dataset combines four sources (CLDR, LOC ISO 639-2, Wikimedia, ISO 639-5) into 858 rows
(special-purpose non-language codes like `und` and `zxx` are excluded at build time).
This notebook answers:
- How many languages are available under different filter strategies?
- What are the non-modern languages — ancient, extinct, constructed?
- How does coverage vary by script, family, and directionality?

In [1]:
import os
import sys
import pandas as pd
import altair as alt
from pathlib import Path

alt.data_transformers.enable("vegafusion")

sys.path.insert(0, str(Path("..").resolve()))
from scripts.utils import get_data_directory_path

DATA_DIR = get_data_directory_path()
CSV_PATH = os.path.join(DATA_DIR, "metadata_files", "language_codes_comprehensive.csv")

df = pd.read_csv(CSV_PATH)
print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

Total rows: 858
Columns: ['language_code', 'language_name', 'local_name', 'iso639_1', 'iso639_2_t', 'iso639_2_b', 'scripts', 'script_codes', 'primary_script', 'primary_script_code', 'directionality', 'directionality_wikimedia', 'modern_language', 'is_official', 'n_scripts', 'in_iso639_1', 'in_iso639_2', 'in_wikimedia', 'is_group_iso639_2', 'sources', 'iso639_5_direct', 'iso639_5_family', 'family_name', 'subfamily_name']


In [2]:
import json
from pathlib import Path
from scripts.experiment.generate_language_codes import MANUAL_LANG_TO_SET5

_json_path = Path("..") / "datasets" / "metadata_files" / "language_family_assignments.json"
with open(_json_path, encoding="utf-8") as _f:
    _json_entries = {e["language_code"]: e for e in json.load(_f)}

def _family_source(row):
    code = str(row["language_code"]).strip()
    iso2 = str(row.get("iso639_2_t", "")).strip()
    fname = row["family_name"]
    if pd.isna(fname) or str(fname).strip() == "":
        return "Unassigned"
    if MANUAL_LANG_TO_SET5.get(code) or MANUAL_LANG_TO_SET5.get(iso2):
        return "Manual mapping"
    if code in _json_entries:
        return "JSON assessment"
    return "Unassigned"

df["family_name_source"] = df.apply(_family_source, axis=1)
print(df["family_name_source"].value_counts().to_string())

family_name_source
JSON assessment    610
Manual mapping     246
Unassigned           2


In [3]:
df[df.family_name_source == "Unassigned"]

,language_code,language_name,local_name,iso639_1,iso639_2_t,iso639_2_b,scripts,script_codes,primary_script,primary_script_code,...,in_iso639_1,in_iso639_2,in_wikimedia,is_group_iso639_2,sources,iso639_5_direct,iso639_5_family,family_name,subfamily_name,family_name_source
516,NaN,Min Nan Chinese,NaN,NaN,NaN,NaN,Simplified,Hans,Simplified,Hans,...,False,False,False,False,cldr,NaN,NaN,NaN,NaN,Unassigned
658,see also: test languages at the Wikimedia Incu...,see also: test languages at the Wikimedia Incu...,see also: test languages at the Wikimedia Incu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,True,False,wikimedia_only,NaN,NaN,NaN,NaN,Unassigned


## 1.1 Source Overlap

How many languages come from each source, and how much do they overlap?

In [4]:
source_flags = {
    "CLDR":      df["sources"].str.contains("cldr", na=False),
    "LOC ISO 639-2": df["in_iso639_2"].fillna(False).astype(bool),
    "Wikimedia": df["in_wikimedia"].fillna(False).astype(bool),
}

summary = pd.DataFrame({
    "source": list(source_flags.keys()),
    "count":  [m.sum() for m in source_flags.values()],
})

bars = alt.Chart(summary).mark_bar().encode(
    y=alt.Y("source:N", sort="-x", title=None),
    x=alt.X("count:Q", title="number of language codes"),
    color=alt.Color("source:N", legend=None, scale=alt.Scale(scheme="tableau10")),
    tooltip=["source:N", "count:Q"],
).properties(width=380, height=120, title="Language codes per source")

text = bars.mark_text(align="left", dx=4, fontSize=10).encode(
    text="count:Q", color=alt.value("black"))

# Overlap: in all three, in two, in one
df["n_sources"] = (
    source_flags["CLDR"].astype(int) +
    source_flags["LOC ISO 639-2"].astype(int) +
    source_flags["Wikimedia"].astype(int)
)
overlap = df["n_sources"].value_counts().reset_index()
overlap.columns = ["n_sources", "count"]
overlap["label"] = overlap["n_sources"].map({1: "1 source only", 2: "2 sources", 3: "all 3 sources"})

pie = alt.Chart(overlap).mark_arc(innerRadius=40).encode(
    theta="count:Q",
    color=alt.Color("label:N", title="overlap", scale=alt.Scale(scheme="set2")),
    tooltip=["label:N", "count:Q"],
).properties(width=200, height=200, title="Source overlap")

(bars + text) | pie

alt.HConcatChart(...)

## 1.2 Filter Strategy Comparison

Four candidate target sets, from conservative to broad.

In [5]:
filters = {
    "All 858 (pipeline default)":       pd.Series([True] * len(df)),
    "Wikimedia only":                    df["in_wikimedia"].fillna(False),
    "Modern + ISO 639-2 or Wikimedia":  df["modern_language"] & (df["in_iso639_2"] | df["in_wikimedia"]),
    "All modern":                        df["modern_language"].fillna(False),
}

rows = []
for label, mask in filters.items():
    subset = df[mask]
    rows.append({
        "filter": label,
        "total":  len(subset),
        "ltr":    (subset["directionality"] == "ltr").sum(),
        "rtl":    (subset["directionality"] == "rtl").sum(),
        "iso639_1": subset["in_iso639_1"].sum(),
        "iso639_2": subset["in_iso639_2"].sum(),
        "wikimedia": subset["in_wikimedia"].sum(),
    })
filter_df = pd.DataFrame(rows)
filter_df

,filter,total,ltr,rtl,iso639_1,iso639_2,wikimedia
0,All 858 (pipeline default),858,780,78,187,416,270
1,Wikimedia only,270,251,19,185,221,270
2,Modern + ISO 639-2 or Wikimedia,432,403,29,178,383,259
3,All modern,804,735,69,178,383,259


In [6]:
filter_order = filter_df["filter"].tolist()

bar = alt.Chart(filter_df).mark_bar().encode(
    y=alt.Y("filter:N", sort=filter_order, title=None),
    x=alt.X("total:Q", title="languages in target set"),
    color=alt.Color("filter:N", legend=None, scale=alt.Scale(scheme="tableau10")),
    tooltip=["filter:N", "total:Q", "ltr:Q", "rtl:Q", "wikimedia:Q"],
).properties(width=420, height=160, title="Target set size by filter strategy")

text = bar.mark_text(align="left", dx=4, fontSize=10).encode(
    text="total:Q", color=alt.value("black"))

bar + text

alt.LayerChart(...)

In [7]:
# Stacked LTR/RTL breakdown
dir_rows = []
for _, r in filter_df.iterrows():
    dir_rows.append({"filter": r["filter"], "direction": "LTR", "count": r["ltr"]})
    dir_rows.append({"filter": r["filter"], "direction": "RTL", "count": r["rtl"]})
dir_df = pd.DataFrame(dir_rows)

dir_bar = alt.Chart(dir_df).mark_bar().encode(
    y=alt.Y("filter:N", sort=filter_order, title=None),
    x=alt.X("count:Q", title="languages"),
    color=alt.Color("direction:N",
        scale=alt.Scale(domain=["LTR", "RTL"], range=["#1976d2", "#e64a19"]),
        title="direction"),
    tooltip=["filter:N", "direction:N", "count:Q"],
).properties(width=420, height=160, title="LTR / RTL split by filter strategy")

dir_bar

alt.Chart(...)

## 1.3 Languages Added by Each Expansion Step

What languages would we gain (or lose) moving from one filter to the next?

In [8]:
wiki_mask  = df["in_wikimedia"].fillna(False)
iso2_mask  = df["modern_language"] & (df["in_iso639_2"] | df["in_wikimedia"])
modern_mask = df["modern_language"].fillna(False)

# Languages in ISO2-or-Wikimedia but NOT in Wikimedia-only
iso2_additions = df[iso2_mask & ~wiki_mask][
    ["language_code", "language_name", "directionality", "family_name", "sources"]
].sort_values("language_name").reset_index(drop=True)

# Languages in All-Modern but NOT in ISO2-or-Wikimedia
modern_additions = df[modern_mask & ~iso2_mask][
    ["language_code", "language_name", "directionality", "family_name", "sources"]
].sort_values("language_name").reset_index(drop=True)

print(f"Added by expanding to ISO 639-2 or Wikimedia: {len(iso2_additions)}")
print(f"Added by expanding to all modern: {len(modern_additions)}")
print()

print("=== ISO 639-2 expansion additions (sample) ===")
print(iso2_additions.head(20).to_string(index=False))

Added by expanding to ISO 639-2 or Wikimedia: 173
Added by expanding to all modern: 372

=== ISO 639-2 expansion additions (sample) ===
language_code language_name directionality                     family_name       sources
          ace      Acehnese            ltr          Austronesian languages cldr|iso639_2
          ach         Acoli            ltr          Nilo-Saharan languages cldr|iso639_2
          ada       Adangme            ltr     Niger-Kordofanian languages cldr|iso639_2
          ady        Adyghe            ltr             Caucasian languages cldr|iso639_2
          afh      Afrihili            ltr            Artificial languages      loc_only
          ain          Ainu            ltr                Language isolate cldr|iso639_2
          ale         Aleut            ltr          Eskimo-Aleut languages cldr|iso639_2
          anp        Angika            ltr         Indo-European languages cldr|iso639_2
          arp       Arapaho            ltr North American India

In [9]:
print("=== Additional languages from CLDR-only (all modern, no ISO 639-2 or Wikimedia) ===")
print(modern_additions.to_string(index=False))

=== Additional languages from CLDR-only (all modern, no ISO 639-2 or Wikimedia) ===
language_code                  language_name directionality                       family_name sources
          abq                          Abaza            ltr               Caucasian languages    cldr
          abr                          Abron            ltr       Niger-Kordofanian languages    cldr
          agq                          Aghem            ltr       Niger-Kordofanian languages    cldr
          bss                         Akoose            ltr       Niger-Kordofanian languages    cldr
          akz                        Alabama            ltr   North American Indian languages    cldr
          arq                Algerian Arabic            rtl            Afro-Asiatic languages    cldr
          ase         American Sign Language            ltr                    Sign languages    cldr
          amo                            Amo            ltr       Niger-Kordofanian languages    cld

## 1.4 Non-Modern Languages

54 languages flagged as ancient, extinct, or constructed. Are any worth including?
Some (Latin, Sanskrit, Esperanto) have active Wikipedia projects and real scholarly use.

In [10]:
non_modern = df[~df["modern_language"].fillna(True)].copy()
non_modern["has_wikipedia"] = non_modern["in_wikimedia"].fillna(False)
non_modern["has_iso639_1"]  = non_modern["in_iso639_1"].fillna(False)
non_modern["has_iso639_2"]  = non_modern["in_iso639_2"].fillna(False)

display_cols = ["language_code", "language_name", "directionality",
                "has_wikipedia", "has_iso639_1", "has_iso639_2", "family_name"]

print(f"Total non-modern: {len(non_modern)}")
print(f"  With active Wikipedia: {non_modern['has_wikipedia'].sum()}")
print(f"  With ISO 639-1 code:   {non_modern['has_iso639_1'].sum()}")
print()
non_modern[display_cols].sort_values(
    ["has_wikipedia", "has_iso639_1"], ascending=False
).reset_index(drop=True)

Total non-modern: 54
  With active Wikipedia: 11
  With ISO 639-1 code:   9



,language_code,language_name,directionality,has_wikipedia,has_iso639_1,has_iso639_2,family_name
0,cu,Church Slavic,ltr,True,True,True,Indo-European languages
1,eo,Esperanto,ltr,True,True,True,Artificial languages
2,ia,Interlingua,ltr,True,True,True,Artificial languages
3,ie,Interlingue,ltr,True,True,True,Artificial languages
4,la,Latin,ltr,True,True,True,Indo-European languages
5,pi,Pali,ltr,True,True,True,Indo-European languages
6,sa,Sanskrit,ltr,True,True,True,Indo-European languages
7,vo,Volapük,ltr,True,True,True,Artificial languages
8,ang,Old English,ltr,True,False,True,Indo-European languages
9,arc,Aramaic,rtl,True,False,True,Afro-Asiatic languages


## 1.5 Language Family Distribution

How do languages distribute across families for each filter strategy?

In [11]:
family_rows = []
for label, mask in filters.items():
    subset = df[mask]
    counts = subset["family_name"].fillna("Unknown / unassigned").value_counts().reset_index()
    counts.columns = ["family", "count"]
    counts["filter"] = label
    family_rows.append(counts)
family_long = pd.concat(family_rows, ignore_index=True)

# Focus on top 15 families across all filters
top_families = (
    family_long.groupby("family")["count"].sum()
    .nlargest(15).index.tolist()
)
plot_df = family_long[family_long["family"].isin(top_families)]

alt.Chart(plot_df).mark_bar().encode(
    x=alt.X("count:Q", title="languages"),
    y=alt.Y("family:N", sort="-x", title=None),
    color=alt.Color("filter:N", sort=list(filters.keys()),
        scale=alt.Scale(scheme="tableau10"), title="filter"),
    xOffset="filter:N",
    tooltip=["filter:N", "family:N", "count:Q"],
).properties(
    width=500, height=420,
    title="Top 15 language families by filter strategy",
)

alt.Chart(...)

## 1.6 Script Distribution

Which scripts are represented, and how does that change across filter strategies?

In [12]:
script_rows = []
for label, mask in filters.items():
    subset = df[mask]
    counts = subset["primary_script"].fillna("Unknown").value_counts().reset_index()
    counts.columns = ["script", "count"]
    counts["filter"] = label
    script_rows.append(counts)
script_long = pd.concat(script_rows, ignore_index=True)

top_scripts = (
    script_long.groupby("script")["count"].sum()
    .nlargest(12).index.tolist()
)
script_plot = script_long[script_long["script"].isin(top_scripts)]

alt.Chart(script_plot).mark_bar().encode(
    x=alt.X("count:Q", title="languages"),
    y=alt.Y("script:N", sort="-x", title=None),
    color=alt.Color("filter:N", sort=list(filters.keys()),
        scale=alt.Scale(scheme="tableau10"), title="filter"),
    xOffset="filter:N",
    tooltip=["filter:N", "script:N", "count:Q"],
).properties(
    width=500, height=380,
    title="Top 12 scripts by filter strategy",
)

alt.Chart(...)

## 1.7 Full Target Set

The pipeline uses all 858 languages — the most defensible and comprehensive choice.
Most ancient/rare languages will produce no translation from direct services, but LLMs
will return results for many of them, which is itself an interesting finding.

The `load_language_codes()` function in `generate_language_codes.py` returns all 858
rows with no additional filtering. Use `in_wikimedia`, `in_iso639_2`, and
`modern_language` columns to filter downstream if needed.

In [13]:
df.directionality.value_counts()

directionality
ltr    780
rtl     78
Name: count, dtype: int64

## 1.8 Family Name Assignment Sources

Where did each language's `family_name` come from?

- **Manual mapping** — hardcoded in `MANUAL_LANG_TO_SET5` (covers major ISO 639-1 / Wikimedia codes; 246 codes)
- **JSON assessment** — assigned via `language_family_assignments.json` (Claude's linguistic assessment for the remaining codes not in MANUAL; 610 codes)
- **Unassigned** — no family name could be determined (2 non-language metadata rows)

Note: an "ISO 639-5 self" category (where a language code is itself a family entry) was considered but fires for only one code (`nah`), which is already caught by the manual mapping.

In [14]:
source_order = ["Manual mapping", "JSON assessment", "Unassigned"]
source_colors = {
    "Manual mapping":  "#1976d2",
    "JSON assessment": "#e64a19",
    "Unassigned":      "#9e9e9e",
}

source_counts = (
    df["family_name_source"].value_counts()
    .reindex(source_order, fill_value=0)
    .reset_index()
)
source_counts.columns = ["source", "count"]
source_counts["pct"] = (source_counts["count"] / len(df) * 100).round(1)
source_counts["label"] = source_counts["count"].astype(str) + " (" + source_counts["pct"].astype(str) + "%)"

# ── Bar: overall source breakdown ────────────────────────────────────────────
bar = alt.Chart(source_counts).mark_bar().encode(
    y=alt.Y("source:N", sort=source_order, title=None),
    x=alt.X("count:Q", title="number of language codes"),
    color=alt.Color(
        "source:N",
        sort=source_order,
        scale=alt.Scale(
            domain=list(source_colors.keys()),
            range=list(source_colors.values()),
        ),
        legend=None,
    ),
    tooltip=["source:N", "count:Q", "pct:Q"],
).properties(width=380, height=120, title="Family name assignment source (all 858 codes)")

text = bar.mark_text(align="left", dx=5, fontSize=10).encode(
    text="label:N", color=alt.value("black"))

# ── Stacked bar: source mix per top family ────────────────────────────────────
top_fams = (
    df[df["family_name"].notna() & (df["family_name"] != "")]
    ["family_name"].value_counts().nlargest(14).index.tolist()
)
fam_src = (
    df[df["family_name"].isin(top_fams)]
    .groupby(["family_name", "family_name_source"])
    .size()
    .reset_index(name="count")
)
fam_order = (
    fam_src.groupby("family_name")["count"].sum()
    .sort_values(ascending=False).index.tolist()
)

stacked = alt.Chart(fam_src).mark_bar().encode(
    y=alt.Y("family_name:N", sort=fam_order, title=None),
    x=alt.X("count:Q", title="languages", stack="normalize",
            axis=alt.Axis(format="%")),
    color=alt.Color(
        "family_name_source:N",
        sort=source_order,
        scale=alt.Scale(
            domain=list(source_colors.keys()),
            range=list(source_colors.values()),
        ),
        title="source",
    ),
    tooltip=["family_name:N", "family_name_source:N", "count:Q"],
).properties(width=380, height=360, title="Source mix by family (top 14, normalised)")

(bar + text) & stacked

alt.VConcatChart(...)

## 1.9 Directionality by Family and Source

For the top 16 families: how many languages are LTR vs RTL, and which source database brought them in?

Source categories:
- **Wikimedia** — has an active Wikipedia project (most complete, highest real-world use)
- **CLDR + ISO 639-2** — in both CLDR and LOC ISO 639-2 but no Wikipedia
- **CLDR only** — in CLDR but not ISO 639-2 or Wikimedia (many low-resource / regional codes)
- **LOC ISO 639-2 only** — archival/historical codes not yet in CLDR
- **Other** — Wikimedia-sourced codes without CLDR coverage

The two panels use independent x-scales because RTL languages are far fewer than LTR in every family.

In [18]:
def _src_cat(row):
    if row.get("in_wikimedia"):
        return "Wikimedia"
    src = str(row.get("sources", "") or "")
    if "loc_only" in src:
        return "LOC ISO 639-2 only"
    if "cldr" in src and row.get("in_iso639_2"):
        return "CLDR + ISO 639-2"
    if "cldr" in src:
        return "CLDR only"
    return "Other"

df["source_cat"] = df.apply(_src_cat, axis=1)

top_fams = df["family_name"].dropna().value_counts().nlargest(16).index.tolist()

dir_rows = []
for (family, direction, source), grp in (
    df[df["family_name"].isin(top_fams)]
    .groupby(["family_name", "directionality", "source_cat"])
):
    dir_rows.append({
        "family": family,
        "direction": direction.upper(),
        "source": source,
        "count": len(grp),
    })
dir_src_df = pd.DataFrame(dir_rows)

SOURCE_ORDER = ["Wikimedia", "CLDR + ISO 639-2", "CLDR only", "LOC ISO 639-2 only", "Other"]
SOURCE_COLORS = ["#1976d2", "#e64a19", "#f57c00", "#388e3c", "#9e9e9e"]

fam_order = (
    dir_src_df.groupby("family")["count"].sum()
    .sort_values(ascending=False).index.tolist()
)

alt.Chart(dir_src_df).mark_bar().encode(
    y=alt.Y("family:N", sort=fam_order, title=None),
    x=alt.X("count:Q", title="number of languages"),
    color=alt.Color(
        "source:N",
        sort=SOURCE_ORDER,
        scale=alt.Scale(domain=SOURCE_ORDER, range=SOURCE_COLORS),
        title="Source",
    ),
    tooltip=["family:N", "direction:N", "source:N", "count:Q"],
).facet(
    column=alt.Column(
        "direction:N",
        sort=["LTR", "RTL"],
        title=None,
        header=alt.Header(labelFontWeight="bold", labelFontSize=13),
    ),
).resolve_scale(x="independent").properties(
    title=alt.Title(
        "Directionality by language family (top 16), colored by source",
        subtitle="Independent x-scales — RTL languages are far fewer in every family",
    )
)

alt.FacetChart(...)